# GfE ↔ Load Joint Toy v2 (P0–P2 refined)

**Purpose.** Go/no-go test of the Stage-1 ↔ Stage-2/3 workflow:

> Observation-channel computational entropy + load (Stage 1) and  
> Bianconi GfE relative entropy on an induced metric (Stages 2–3)  
> describe the same structure-preserving process on a minimal 1D system.

**This version (vs v1):**
- **P0:** Classical Perona–Malik conductivity with noise-scale \(K\) (no staircasing ramps)
- **P1:** Observation-channel \(H_c\): residual MSE + edge-location posterior entropy
- **P2:** Split load \(L_E, L_S, L_B\); clock from \(L_E + L_S\)
- **Scorecard v2:** residual dual, edge retention, ramp stability, \(L_E\leftrightarrow G\), load clock, residual co-motion

**Implementation:** core logic in [`_joint_toy_v2_core.py`](_joint_toy_v2_core.py) (shared with CLI).

**Docs:** [DESIGN](DESIGN_gfe_load_joint_toy.md) · [PRD](../../PRD.md) · [bridge](../../synthesis/bridge-bianconi-relative-entropy.md)

**Kernel:** select **Python (computational-entropy)** / project `.venv`.



In [ ]:
# Setup
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Ensure bridging dir is on path when run from notebook or other cwd
HERE = Path.cwd()
if (HERE / "_joint_toy_v2_core.py").exists():
    sys.path.insert(0, str(HERE))
elif (HERE / "simulations" / "bridging" / "_joint_toy_v2_core.py").exists():
    sys.path.insert(0, str(HERE / "simulations" / "bridging"))
else:
    # notebook often opened from repo root or bridging/
    for p in [HERE, HERE.parent, Path("simulations/bridging")]:
        if (p / "_joint_toy_v2_core.py").exists():
            sys.path.insert(0, str(p.resolve()))
            break

import _joint_toy_v2_core as toy

np.random.seed(42)
plt.rcParams.update({
    "figure.figsize": (10, 3.5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

N = toy.N
x = np.arange(N, dtype=float)
print(f"N={N}, ALPHA_G={toy.ALPHA_G}, K_PM={toy.K_PM}, ALPHA_L={toy.ALPHA_L}")
print(f"dt={toy.DT}, steps={toy.N_STEPS}, noise={toy.NOISE_SIGMA}")



## 1. Hidden structure \(\phi_\star\) and noisy observations (Stage 1 channel)

1. Hidden clean \(\phi_\star\)
2. Observation \(y = \phi_\star + \eta\)
3. Dynamics reconstructor \(\hat\phi(t)\) starts from \(y\)
4. \(H_c\) from residual + edge-location entropy (not histogram of \(\hat\phi\))



In [ ]:
ICS = toy.build_ics()
fig, axes = plt.subplots(1, 4, figsize=(13, 2.8))
for ax, (name, pack) in zip(axes, ICS.items()):
    ax.plot(x, pack["star"], "k-", lw=1.5, label=r"$\phi_\star$")
    ax.plot(x, pack["y"], color="C1", alpha=0.7, lw=1.0, label=r"$y$")
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("x")
axes[0].legend(fontsize=8)
axes[0].set_ylabel(r"$\phi$")
fig.suptitle("Hidden structure vs observation", y=1.03)
plt.tight_layout()
plt.show()



## 2. Induced metric \(G\) and GfE action (Stages 2–3)

\[
G = 1 + \alpha_G(\nabla\phi)^2,\qquad S_{\mathrm{GfE}} = -\sum \ln G.
\]



In [ ]:
pack0 = ICS["noisy_step"]
phi0 = pack0["y"]
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
axes[0].plot(x, pack0["star"], "k--", alpha=0.5, label="star")
axes[0].plot(x, phi0, color="C1", label="y")
axes[0].legend(fontsize=8); axes[0].set_title(r"$\phi$")
axes[1].plot(x[:-1] + 0.5, toy.induced_G(phi0) - 1, color="C2")
axes[1].set_title(r"Induced $G-1$")
axes[2].plot(x[:-1] + 0.5, toy.gfe_density(phi0), color="C3")
axes[2].set_title(r"GfE density $-\ln G$")
for ax in axes:
    ax.set_xlabel("x")
plt.tight_layout()
plt.show()
print(f"S_GfE(y)={toy.gfe_action(phi0):.4f}, residual MSE={toy.residual_mse(phi0, pack0['star']):.4f}")
print(f"H_c(star)={toy.H_c_channel(pack0['star'], pack0['star']):.4f}, H_c(y)={toy.H_c_channel(phi0, pack0['star']):.4f}")



## 3. Split load and dynamics

- **Heat** · **PM** (\(K=K_{PM}\)) · **Load-gated PM** with \(dt/(1+\alpha_L(L_E+L_S))\)



In [ ]:
MODES = ["heat", "pm", "load_pm"]
MODE_LABELS = {
    "heat": "Isotropic heat",
    "pm": "PM / GfE",
    "load_pm": r"Load-gated PM ($L_E+L_S$)",
}
COLORS = {"heat": "C3", "pm": "C0", "load_pm": "C2"}

ICS, results = toy.run_all(ICS)
for ic_name, pack in ICS.items():
    for mode in MODES:
        r = results[ic_name][mode]
        print(f"{ic_name:16s} {mode:8s}  resid {r['residual'][0]:.4f}->{r['residual'][-1]:.4f}  "
              f"H_c {r['H_c'][0]:.3f}->{r['H_c'][-1]:.3f}  "
              f"max|g| {r['max_grad'][0]:.3f}->{r['max_grad'][-1]:.3f}")



## 4. Final reconstructor vs truth



In [ ]:
def plot_final(ic_name):
    pack = ICS[ic_name]
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
    axes[0].plot(x, pack["star"], "k-", lw=2, label=r"$\phi_\star$")
    axes[0].plot(x, pack["y"], color="gray", alpha=0.4, lw=1, label="y")
    for mode in MODES:
        axes[0].plot(x, results[ic_name][mode]["phi"][-1], color=COLORS[mode], label=MODE_LABELS[mode])
    axes[0].set_title(rf"$\hat\phi$ final — {ic_name}")
    axes[0].legend(fontsize=7)

    for mode in MODES:
        phi_f = results[ic_name][mode]["phi"][-1]
        axes[1].plot(x[:-1] + 0.5, toy.induced_G(phi_f) - 1, color=COLORS[mode], label=mode)
    axes[1].set_title(r"Induced $G-1$ (final)")
    axes[1].legend(fontsize=8)

    for mode in MODES:
        phi_f = results[ic_name][mode]["phi"][-1]
        axes[2].plot(x, np.abs(phi_f - pack["star"]), color=COLORS[mode], label=mode)
    axes[2].set_title(r"$|\hat\phi-\phi_\star|$")
    axes[2].legend(fontsize=8)
    fig.suptitle(ICS[ic_name]["desc"], y=1.02)
    plt.tight_layout()
    plt.show()

for name in ICS:
    plot_final(name)



## 5. Channel diagnostics (residual, \(H_c\), split load, edges)



In [ ]:
def plot_diagnostics(ic_name):
    fig, axes = plt.subplots(2, 3, figsize=(13, 6.5))
    specs = [
        (0, 0, "residual", "Residual MSE"),
        (0, 1, "H_c", r"Channel $H_c$"),
        (0, 2, "max_grad", r"$\max|\nabla\hat\phi|$"),
        (1, 0, "L_E", r"$L_E$ induction"),
        (1, 1, "L_S", r"$L_S$ entropy rate"),
        (1, 2, "S_gfe", r"$S_{\mathrm{GfE}}$"),
    ]
    for i, j, key, title in specs:
        ax = axes[i, j]
        for mode in MODES:
            r = results[ic_name][mode]
            ax.plot(r["t"], r[key], color=COLORS[mode], label=MODE_LABELS[mode])
        ax.set_title(title)
        ax.set_xlabel("t")
        ax.legend(fontsize=6)
    fig.suptitle(f"Diagnostics — {ic_name}", y=1.01)
    plt.tight_layout()
    plt.show()

    r = results[ic_name]["load_pm"]
    fig, ax = plt.subplots(figsize=(10, 2.0))
    ax.plot(r["t"], r["clock"], color="C2")
    ax.set_ylabel(r"$1/(1+\alpha_L L_{clock})$")
    ax.set_xlabel("t")
    ax.set_title(f"Effective clock — {ic_name}")
    plt.tight_layout()
    plt.show()

for name in ICS:
    plot_diagnostics(name)



## 6. \(L_E\) vs induction intensity \(\mathbb{E}[G-1]\)



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ic = "noisy_step"
for mode in MODES:
    r = results[ic][mode]
    axes[0].scatter(r["mean_G_minus_1"], r["L_E"], s=20, color=COLORS[mode], label=mode)
    axes[1].scatter(r["mean_G_minus_1"], r["L"], s=20, color=COLORS[mode], label=mode)
axes[0].set_xlabel(r"$\mathbb{E}[G-1]$"); axes[0].set_ylabel(r"$L_E$")
axes[0].set_title(f"{ic}: $L_E$ vs induction")
axes[1].set_xlabel(r"$\mathbb{E}[G-1]$"); axes[1].set_ylabel(r"$L$ total")
axes[1].set_title(f"{ic}: $L$ total vs induction")
axes[0].legend(fontsize=8); axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()



## 7. Scorecard v2



In [ ]:
card = toy.evaluate(ICS, results)
toy.print_scorecard(card)



## 8. Summary figure + interpretation

| Stage | Objects | What to look for |
|-------|---------|------------------|
| 1 | residual, \(H_c\), \(L_E/L_S/L_B\), clock | PM reduces residual vs heat; load slows clocks |
| 2 | \(G-1\) | Peaks at edges under PM |
| 3 | \(S_{\mathrm{GfE}}\), PM flow | Structure-preserving denoising |

**If ≥5/6 SUPPORT:** Euclidean warm-up dual is promising → Action–Channel Duality note.  
**If MIXED:** geometry/clock hold; channel dual partial.  
**If WEAK:** keep GfE as continuum peer only.



In [ ]:
ic = "noisy_step"
pack = ICS[ic]
fig, axes = plt.subplots(2, 2, figsize=(10, 6))
axes[0, 0].plot(x, pack["star"], "k-", lw=2, label=r"$\phi_\star$")
axes[0, 0].plot(x, pack["y"], color="gray", alpha=0.35, label="y")
for mode in MODES:
    axes[0, 0].plot(x, results[ic][mode]["phi"][-1], color=COLORS[mode], label=MODE_LABELS[mode])
axes[0, 0].set_title(r"$\hat\phi$ final"); axes[0, 0].legend(fontsize=7)

for mode in MODES:
    r = results[ic][mode]
    axes[0, 1].plot(r["t"], r["residual"], color=COLORS[mode], label=mode)
axes[0, 1].set_title("Residual MSE"); axes[0, 1].legend(fontsize=7)

for mode in MODES:
    r = results[ic][mode]
    axes[1, 0].plot(r["t"], r["H_c"], color=COLORS[mode], label=mode)
axes[1, 0].set_title(r"Channel $H_c$"); axes[1, 0].legend(fontsize=7)

for mode in MODES:
    r = results[ic][mode]
    axes[1, 1].plot(r["t"], r["max_grad"], color=COLORS[mode], label=mode)
axes[1, 1].set_title(r"$\max|\nabla\hat\phi|$"); axes[1, 1].legend(fontsize=7)

fig.suptitle("Joint toy v2 summary — noisy_step (P0–P2)", y=1.01)
plt.tight_layout()
# save next to core module if possible
out = Path("_joint_toy_v2_core.py").resolve().parent / "gfe_load_joint_toy_summary.png"
if not Path("_joint_toy_v2_core.py").exists():
    out = Path("simulations/bridging/gfe_load_joint_toy_summary.png")
    out.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out, dpi=140, bbox_inches="tight")
print("Saved", out)
print(f"Scorecard: {card['support']}/6 — {card['verdict']}")
plt.show()

